## Plug-and-Play Checkpoints: Reproduce Table 1 (NovoMolGen-32M)

> *Run this single cell in the notebook; it downloads the checkpoint, samples
> 30000 valid/canonical SMILES, evaluates the six metrics used in Table 1, and
> renders the result as a tidy dataframe.*

In [1]:
import os
from pathlib import Path

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HOME"] = "/network/scratch/k/kamran.chitsaz/hf_home"
os.chdir(Path(os.getcwd()).parent)

from transformers import AutoTokenizer, AutoModelForCausalLM
from accelerate import Accelerator
import rootutils

rootutils.setup_root(os.getcwd(), indicator=".project-root", pythonpath=True)

from src.eval import MoleculeEvaluator
from src.models import generate_valid_smiles, prepare_hf_model
from src.data_loader.utils import load_valid_and_test_data


There was a problem when trying to write in your cache folder (/network/scratch/k/kamran.chitsaz/hf_home/hub). You should set the environment variable TRANSFORMERS_CACHE to a writable directory.
2025-09-23 14:41:54.827 | WARNING  | src.eval.molecule_evaluation:<module>:18 - Failed to import reactivity: xtb C extension unimportable, cannot use C-API
2025-09-23 14:41:55.274 | WARNING  | src.eval.molecule_evaluation:<module>:26 - Failed to import tadf: 'XTBHOME'
Failed to find the pandas get_adjustment() function to patch
Failed to patch pandas - PandasTools will have limited functionality


In [ ]:
# 1.  Load pretrained 32 M checkpoint + tokenizer
tokenizer = AutoTokenizer.from_pretrained("chandar-lab/NovoMolGen_32M_SMILES_AtomWise", trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained("chandar-lab/NovoMolGen_32M_SMILES_AtomWise", trust_remote_code=True)

acc = Accelerator(mixed_precision='bf16')
model = acc.prepare(model)

# 2.  Load pre-computed dataset stats
df_test, df_valid = load_valid_and_test_data("MolGen/ZINC_1B-raw", "175k")
task = MoleculeEvaluator(
    task_names=['unique@1k', 'IntDiv', 'FCD', 'Frag', 'Scaf', 'SNN'],  
    valid_stats=df_valid, 
    test_stats=df_test,
    n_jobs=2
    )

# 3.  Sample 30000 molecules
smiles_list = []
for _ in range(10):
    outputs = model.sample(
        tokenizer=tokenizer, 
        batch_size=3000, 
        max_length=64, 
        temperature=1.0, 
        top_k=0, 
        top_p=0, 
        )
    smiles_list.extend(outputs['SMILES'])

# 4.  Evaluate & display
metrics = task(smiles_list, filter=True, return_valid_index=True)

import pandas as pd
from IPython.display import display

row = {
    "Validity": metrics["validity"],
    "Unique@1k": metrics["unique@1k"],
    "IntDiv":    metrics["IntDiv"],
    "FCD":       metrics["FCD"]["FCD"],
    "SNN":       metrics["SNN"]["SNN"],
    "Frag":      metrics["Frag"]["Frag"],
    "Scaf":      metrics["Scaf"]["Scaf"],
}

df = pd.DataFrame([row]).round(4)
display(df.style.hide(axis="index").set_caption("NovoMolGen-32 M ─ Table 1 metrics"))

tokenizer_config.json:   0%|          | 0.00/943 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/555 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/863 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/126M [00:00<?, ?B/s]

OSError: Can't load the model for 'chandar-lab/NovoMolGen_32M_SMILES_AtomWise'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'chandar-lab/NovoMolGen_32M_SMILES_AtomWise' is the correct path to a directory containing a file named pytorch_model.bin, tf_model.h5, model.ckpt or flax_model.msgpack.

### HF Checkpoint

In [3]:
# 1.  Load HF checkpoint (no flash attention)
model = AutoModelForCausalLM.from_pretrained("chandar-lab/NovoMolGen_32M_SMILES_AtomWise", revision='hf-checkpoint', device_map='auto')
tokenizer = AutoTokenizer.from_pretrained("chandar-lab/NovoMolGen_32M_SMILES_AtomWise", revision='hf-checkpoint')
model = prepare_hf_model(model)

# 2.  Load pre-computed dataset stats
df_test, df_valid = load_valid_and_test_data("MolGen/ZINC_1B-raw", "175k")
task = MoleculeEvaluator(
    task_names=['unique@1k', 'IntDiv', 'FCD', 'Frag', 'Scaf', 'SNN'],  
    valid_stats=df_valid, 
    test_stats=df_test,
    n_jobs=2
    )

# 3.  Sample 30000 molecules
smiles_list = []
for _ in range(10):
    outputs = model.sample(
        tokenizer=tokenizer, 
        batch_size=1000, 
        max_length=64, 
        temperature=1.0, 
        top_k=0, 
        top_p=1.0,
        do_sample=True, 
        )
    smiles_list.extend(outputs['SMILES'])

# 4.  Evaluate & display
metrics = task(smiles_list, filter=True, return_valid_index=True)

import pandas as pd
from IPython.display import display

row = {
    "Validity": metrics["validity"],
    "Unique@1k": metrics["unique@1k"],
    "IntDiv":    metrics["IntDiv"],
    "FCD":       metrics["FCD"]["FCD"],
    "SNN":       metrics["SNN"]["SNN"],
    "Frag":      metrics["Frag"]["Frag"],
    "Scaf":      metrics["Scaf"]["Scaf"],
}

df = pd.DataFrame([row]).round(4)
display(df.style.hide(axis="index").set_caption("NovoMolGen-32 M ─ Table 1 metrics"))

PermissionError: [Errno 13] Permission denied: '/network'